# RGT-Est on Google Colab: horizon-guided vs unguided (Netherlands F3)

Runs the pretrained RGT-Est model (Dou, Wu, Gao & Bi, 2026) twice on the F3 survey, which comes with interpreted horizons:

- **Unguided:** seismic only.
- **Guided:** seismic plus a subset of the interpreted horizons.

To keep the comparison fair, the horizons are split. **Every second horizon is used as guidance, and the others are held back** and used only for scoring. Both runs are scored on the held-back horizons, which the model never sees.

**Before you start**
1. **Runtime → Change runtime type → GPU.** An **L4** is recommended, because the T4 ran out of memory on Poseidon.
2. Open the **Files** panel (folder icon on the left) and drag these into the top-level `content` folder:
   - `RGT-Est_CIG-Benchmark.pt` (the pretrained model)
   - `F3_hoirz_seis_fault.npz` (from the `RealData` folder of the Zenodo download)
3. **Runtime → Run all.**

Files in the Colab session are deleted when the runtime disconnects or changes type.

## 1. Settings

In [ ]:
DATA_FOLDER  = "/content"                    # where you uploaded the files
MODEL_FILE   = "RGT-Est_CIG-Benchmark.pt"
DATA_FILE    = "F3_hoirz_seis_fault.npz"

GUIDE_EVERY  = 2      # use every 2nd horizon as guidance; the rest are held back for scoring.
                      # 3 = every 3rd (less guidance), 1 = all horizons (nothing held back to score).
TOLERANCE    = 2      # a trace counts as a "hit" if the predicted horizon is within this many samples

## 2. Check the GPU and the uploaded files

In [ ]:
import os, glob, torch
print("PyTorch", torch.__version__)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)")
else:
    print("No GPU found. Go to Runtime → Change runtime type → GPU, then run again.")

def find_file(name):
    p = os.path.join(DATA_FOLDER, name)
    if os.path.exists(p):
        return p
    hits = glob.glob(f"/content/**/{name}", recursive=True)   # also look in subfolders
    return hits[0] if hits else None

paths = {}
for f in (MODEL_FILE, DATA_FILE):
    p = find_file(f)
    print(f"found    {p}  ({os.path.getsize(p)/1e6:.0f} MB)" if p else f"MISSING  {f}")
    paths[f] = p
if None in paths.values():
    raise FileNotFoundError("Upload the missing file(s) into the Files panel (left sidebar), "
                            "or fix the names in the Settings cell, then run this cell again.")

## 3. Get the RGT-Est code (your fork)

In [ ]:
!git clone -q https://github.com/fergusrssmith/RGT /content/RGT 2>/dev/null || git -C /content/RGT pull -q
import sys
sys.path.insert(0, "/content/RGT/demo")

## 4. Load the seismic and horizons, and split the horizons

The crop (dropping the first 50 time samples and keeping the first 580 inlines) is the same as the authors' demo.

In [ ]:
import numpy as np

def normalization(data):
    rng = np.max(data) - np.min(data)
    return (data - np.min(data)) / (rng + 1e-6)

def z_score_clip(data, clp_s=2.0):
    z = (data - np.mean(data)) / (np.std(data) + 1e-8)
    return normalization(np.clip(z, -clp_s, clp_s))

data = np.load(paths[DATA_FILE])
print("arrays in file:", {k: data[k].shape for k in data.files})

seis_raw = data["seis"][50:, :580].astype(np.float32)      # (time, inline, xline)
horiz_id = data["horiz"][50:, :580].astype(np.int32)       # 0 = no horizon, otherwise a horizon ID
T, H, W = seis_raw.shape
seis = z_score_clip(seis_raw) * 2 - 1                      # [-1, 1], (time, inline, xline)
print("cropped shape (time, inline, xline):", seis.shape)

ids = np.unique(horiz_id)[1:]
# Sort the horizons shallow → deep by their average time
ids = ids[np.argsort([np.argwhere(horiz_id == i)[:, 0].mean() for i in ids])]
guide_ids = ids[::GUIDE_EVERY]
test_ids  = np.setdiff1d(ids, guide_ids) if GUIDE_EVERY > 1 else ids
print(f"{len(ids)} horizons: {len(guide_ids)} used as guidance, {len(test_ids)} held back for scoring")

## 5. Run the model: unguided and guided

The guidance follows the authors' demo. Each guidance horizon is thickened slightly and given a fixed RGT value from its average time, and the model also receives a mask showing where those horizons are.

In [ ]:
import time
import torch.nn as nn
import torch.nn.functional as F
from scipy.ndimage import grey_dilation

device = "cuda" if torch.cuda.is_available() else "cpu"
model = torch.jit.load(paths[MODEL_FILE], map_location=device).eval()
INFER_SHAPE = (400, 512, 512)

def guidance_volume(use_ids):
    vol = np.zeros((T, H, W), np.float32)
    if len(use_ids) == 0:
        return vol
    sel = np.where(np.isin(horiz_id, use_ids), horiz_id, 0)
    sel = grey_dilation(sel, size=2)
    for i in use_ids:
        rgt_value = np.argwhere(horiz_id == i)[:, 0].mean() / T
        vol[sel == i] = rgt_value * 2 - 1
    return vol

def run_rgt(guide):
    s = torch.from_numpy(seis)[None, None]
    g = torch.from_numpy(guide)[None, None]
    x = torch.cat([s, g, (g != 0).float()], dim=1)
    x = F.interpolate(x, INFER_SHAPE, mode="nearest").to(device)
    t0 = time.time()
    with torch.no_grad(), torch.autocast(device_type=device):
        rgt = model(nn.ReflectionPad3d(8)(x))[:, :, 8:-8, 8:-8, 8:-8]
    print(f"  inference took {time.time()-t0:.1f} s")
    out = F.interpolate(rgt.float(), (T, H, W), mode="trilinear").cpu().numpy()[0, 0]
    del x, rgt
    if device == "cuda": torch.cuda.empty_cache()
    return z_score_clip(out)       # (time, inline, xline), values 0–1

guide_vol = guidance_volume(guide_ids)
print("Unguided run:")
rgt_unguided = run_rgt(np.zeros_like(guide_vol))
print("Guided run:")
rgt_guided = run_rgt(guide_vol)

## 6. Score both runs against the held-back horizons

For each held-back horizon, we find the single RGT value that best matches it, trace the predicted horizon at that value, and measure how far it is (in time samples) from the interpreter's horizon on every trace.

- **Mean error:** average vertical miss, in samples. Lower is better.
- **Hit rate:** % of traces where the predicted horizon is within `TOLERANCE` samples. Higher is better.

In [ ]:
import pandas as pd

def score_horizon(rgt, hid, chunk=20000):
    t, i, j = np.nonzero(horiz_id == hid)
    flat = i * W + j
    uniq, inv = np.unique(flat, return_inverse=True)
    t_true = np.bincount(inv, weights=t) / np.bincount(inv)       # interpreted time per trace
    level = np.median(rgt[t, i, j])                                # best-matching RGT value
    ti, tj = uniq // W, uniq % W
    t_pred = np.empty(len(uniq))
    for k in range(0, len(uniq), chunk):                           # time where RGT is closest to that value
        cols = rgt[:, ti[k:k+chunk], tj[k:k+chunk]]
        t_pred[k:k+chunk] = np.abs(cols - level).argmin(axis=0)
    err = np.abs(t_pred - t_true)
    return err.mean(), 100 * (err <= TOLERANCE).mean(), len(uniq), t_true.mean()

rows = []
for hid in ids:
    role = "guidance" if (hid in guide_ids and GUIDE_EVERY > 1) else "held back"
    mu, hu_, n, depth = score_horizon(rgt_unguided, hid)
    mg, hg, _, _ = score_horizon(rgt_guided, hid)
    rows.append(dict(horizon=int(hid), role=role, avg_time=round(depth), traces=n,
                     unguided_err=mu, guided_err=mg, unguided_hit=hu_, guided_hit=hg))
scores = pd.DataFrame(rows)

held = scores[scores.role == "held back"]
print(f"HELD-BACK horizons (the fair test), {len(held)} horizons:")
print(f"  mean error   unguided {held.unguided_err.mean():5.2f} samples   guided {held.guided_err.mean():5.2f} samples")
print(f"  hit rate     unguided {held.unguided_hit.mean():5.1f} %         guided {held.guided_hit.mean():5.1f} %")
if GUIDE_EVERY > 1:
    gd = scores[scores.role == "guidance"]
    print(f"\nGUIDANCE horizons (how well the guided run honours its inputs), {len(gd)} horizons:")
    print(f"  mean error   unguided {gd.unguided_err.mean():5.2f} samples   guided {gd.guided_err.mean():5.2f} samples")

pd.options.display.float_format = "{:.2f}".format
scores

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
x = np.arange(len(scores)); wbar = 0.4
for ax, col, label in ((axes[0], "err", "mean error (samples): lower is better"),
                       (axes[1], "hit", f"hit rate within {TOLERANCE} samples (%): higher is better")):
    ax.bar(x - wbar/2, scores[f"unguided_{col}"], wbar, label="unguided", color="#888888")
    ax.bar(x + wbar/2, scores[f"guided_{col}"],   wbar, label="guided",   color="#7b3fbf")
    ax.set_xticks(x)
    ax.set_xticklabels([f"{h}\n{'G' if r == 'guidance' else 'test'}" for h, r in zip(scores.horizon, scores.role)])
    ax.set_xlabel("horizon ID, shallow → deep (G = used as guidance, test = held back)")
    ax.set_title(label); ax.legend()
fig.tight_layout(); plt.show()

## 7. Compare on sections

Each row is one inline, showing **unguided** (left) and **guided** (right). Coloured lines are the predicted horizons (RGT contours).
The interpreter's horizons are drawn on top: **magenta** = used as guidance, **white** = held back. The better the run, the more closely the coloured lines run parallel to the white ones.

In [ ]:
levels = np.linspace(0.02, 0.98, 40)
# Pick 3 inlines spread across the survey that actually contain horizon picks
picked = np.nonzero((horiz_id > 0).sum(axis=(0, 2)) > 0.5 * W)[0]
if len(picked) == 0:
    picked = np.nonzero((horiz_id > 0).any(axis=(0, 2)))[0]
inlines = picked[np.linspace(0, len(picked) - 1, 5, dtype=int)[1:-1]]
guide_mask = np.isin(horiz_id, guide_ids) if GUIDE_EVERY > 1 else np.zeros_like(horiz_id, bool)
test_mask  = np.isin(horiz_id, test_ids)

fig, axes = plt.subplots(len(inlines), 2, figsize=(18, 4.2 * len(inlines)))
for row, il in enumerate(inlines):
    s = seis[:, il, :]
    v = np.percentile(np.abs(s), 99)
    for col, (name, rgt) in enumerate((("unguided", rgt_unguided), ("guided", rgt_guided))):
        ax = axes[row, col]
        ax.imshow(s, cmap="gray", aspect="auto", vmin=-v, vmax=v)
        ax.contour(rgt[:, il, :], levels=levels, cmap="jet", linewidths=1.0)
        gt, gx = np.nonzero(guide_mask[:, il, :]); ax.scatter(gx, gt, s=1.5, c="magenta")
        tt, tx = np.nonzero(test_mask[:, il, :]);  ax.scatter(tx, tt, s=1.5, c="white")
        ax.set_title(f"inline {il}: {name}"); ax.set_xlabel("crossline"); ax.set_ylabel("time sample")
        ax.set_xlim(0, W - 1); ax.set_ylim(T - 1, 0)
fig.tight_layout(); plt.show()

## 8. (Optional) Save and download both RGT volumes

In [ ]:
from google.colab import files
np.savez_compressed("/content/F3_RGT_results.npz", unguided=rgt_unguided, guided=rgt_guided,
                    guide_ids=guide_ids, test_ids=test_ids)
files.download("/content/F3_RGT_results.npz")